# Stage 5 — Evaluation

**Goal:** measure what the base model can do, so that after instruction tuning
you can answer "did that help?" with evidence instead of vibes.

### Three measurements, and what each is honestly worth

**Perplexity** — `exp(mean cross-entropy)`. Loosely: how many tokens the model is
effectively choosing between at each step. It is the only number here comparable
across checkpoints *of this model*, which is why we track it.

It is **not** comparable across models with different tokenizers. A model with a
smaller vocabulary gets a lower perplexity for free, because it's picking from
fewer options. This is exactly why leaderboards stopped reporting it.

**Repetition metrics** — the failure mode small models actually have. When a model
falls into a loop, perplexity barely moves, because each repeated token is
individually very predictable. This is the cleanest example in the project of a
metric looking healthy while the output is unusable.

**Reading the output** — the one that actually tells you whether it worked.

### What we deliberately don't do

MMLU, HumanEval, GSM8K and friends would return pure noise on a 15.7M-parameter
model trained on children's stories. Running them to produce a number would be
worse than not running them, because the number would look like information.

In [ ]:
# --- Colab bootstrap -------------------------------------------------------
# Set this to YOUR GitHub repo once; every notebook uses the same cell.
REPO_URL = "https://github.com/pythonstudentiam/e2e_llm_demo.git"

import os, subprocess, sys
from pathlib import Path

REPO = Path("/content/e2e_llm_demo")
WORK = Path("/content/work")          # scratch: data + checkpoints (ephemeral!)
WORK.mkdir(parents=True, exist_ok=True)

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

sys.path.insert(0, str(REPO / "src"))

# Colab ships torch; these are the rest. -q to keep the log readable.
%pip install -q sentencepiece "datasets>=3.0" "transformers>=4.45" "huggingface_hub>=0.30"

# HF token from the Colab Secrets panel (key icon, left sidebar). Name it HF_TOKEN
# and enable notebook access. Never paste a token into a cell.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets")
except Exception as e:
    print(f"No HF_TOKEN yet ({e}). Needed from notebook 04 onward.")

import torch
print(f"torch {torch.__version__} | CUDA {torch.cuda.is_available()} | "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

In [ ]:
# A T4 is expected. Anything without CUDA means Runtime > Change runtime type > T4 GPU.
import torch
assert torch.cuda.is_available(), (
    "No GPU. Runtime > Change runtime type > Hardware accelerator: T4 GPU, then re-run."
)
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name} | compute capability {cap[0]}.{cap[1]} | {mem:.1f} GB")

# Turing (7.5) has no bf16. That is why training uses fp16 + GradScaler.
if cap[0] < 8:
    print("\n-> Pre-Ampere GPU: bf16 unavailable, fp16 autocast + loss scaling it is.")
else:
    print("\n-> Ampere or newer: bf16 would work here and would let you drop the GradScaler.")

In [ ]:
from tinyllm import config
from tinyllm.config import (
    model_cfg, train_cfg, data_cfg, tok_cfg, sft_cfg, gen_cfg, quant_cfg, serve_cfg, hub,
)

print(config.summary())

In [ ]:
from pathlib import Path
import torch, gc, numpy as np
from huggingface_hub import hf_hub_download
from tinyllm.tokenizer import load_sp
from tinyllm.data import load_tokens, ensure_token_file
from tinyllm.train import build_model, load_checkpoint, pull_checkpoint

data_dir, tok_dir = WORK / "data", WORK / "tokenizer"
sp_path = tok_dir / "tokenizer.model"

if not sp_path.exists():
    tok_dir.mkdir(parents=True, exist_ok=True)
    got = hf_hub_download(repo_id=hub.ckpt_repo, filename="tokenizer/tokenizer.model")
    sp_path.write_bytes(Path(got).read_bytes())
sp = load_sp(sp_path)

ckpt = WORK / "checkpoints/latest.pt"
if not ckpt.exists():
    print("pulling the pretrained checkpoint from the Hub...")
    ckpt = pull_checkpoint(local_dir=WORK / "checkpoints")
assert ckpt and Path(ckpt).exists(), "No checkpoint found -- run notebook 04 first."

model = build_model(model_cfg, "cuda")
step, meta = load_checkpoint(Path(ckpt), model)
print(f"loaded checkpoint from step {step:,}")

# Fetches from the Hub if this runtime doesn't have it -- which is the normal
# case, since Colab recycles runtimes between sessions.
val_tokens = load_tokens(ensure_token_file("val.bin", data_dir, hub.ckpt_repo))
print(f"val tokens: {len(val_tokens):,}")

## 5.1 — Perplexity on held-out data

The validation split is the dataset's own, never a slice of train, so this number
can't be inflated by contamination.

In [ ]:
from tinyllm.evaluate import perplexity, bits_per_token
import math

ppl = perplexity(model, val_tokens, n_batches=100)

print(f"  validation loss     {ppl['loss']:.4f}  (+/- {ppl['std_loss']:.4f})")
print(f"  perplexity          {ppl['perplexity']:.2f}")
print(f"  bits per token      {bits_per_token(ppl['loss']):.3f}")
print(f"  measured on         {ppl['n_tokens']:,} tokens")
print()
print(f"  random baseline     loss {math.log(model_cfg.vocab_size):.4f} / "
      f"perplexity {model_cfg.vocab_size:,}")
print(f"  -> the model narrowed {model_cfg.vocab_size:,} options down to "
      f"~{ppl['perplexity']:.0f} effective choices per token")

The "bits per token" framing is worth pausing on: a language model *is* a
compressor. A model at 4 bits/token could encode this corpus in 4 bits per token
where a naive scheme needs `log2(8192) = 13`. Learning and compression are the
same thing viewed from two directions.

## 5.2 — Temperature: the coherence/diversity dial

Same prompt, same seed, five temperatures. Low temperature concentrates
probability on the top choices — coherent but repetitive. High temperature
flattens the distribution — varied but incoherent.

Where the sweet spot sits is a property *of this model*, not a universal
constant, which is why it's measured rather than assumed.

In [ ]:
from tinyllm.evaluate import sampling_sweep

for row in sampling_sweep(model, sp, max_new_tokens=110, seed=0):
    d3 = row.get("distinct_3")
    print(f"=== temperature {row['temperature']}  "
          f"(distinct-3: {d3:.2f})" if d3 else f"=== temperature {row['temperature']}")
    print(row["text"][:400])
    print()

In [ ]:
import matplotlib.pyplot as plt

temps = [0.1, 0.3, 0.5, 0.7, 0.8, 1.0, 1.2, 1.5]
rows = sampling_sweep(model, sp, temperatures=temps, max_new_tokens=150, seed=0)

d1 = [r.get("distinct_1", 0) for r in rows]
d3 = [r.get("distinct_3", 0) for r in rows]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(temps, d1, marker="o", label="distinct-1 (unique words)", color="#4C78A8")
ax.plot(temps, d3, marker="s", label="distinct-3 (unique trigrams)", color="#E45756")
ax.axvline(gen_cfg.temperature, color="gray", linestyle="--",
           label=f"our default ({gen_cfg.temperature})")
ax.set_xlabel("temperature"); ax.set_ylabel("distinct-n ratio")
ax.set_title("Repetition vs temperature")
ax.legend(); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

print("Low distinct-n means looping. Below ~0.6 for distinct-3, output usually")
print("reads as broken -- but note that perplexity would not have told you that.")

## 5.3 — Full report, saved

This gets attached to the model card in stage 7, and becomes the *before* half of
the before/after comparison in stage 6.

In [ ]:
from tinyllm.evaluate import full_report, print_report, save_report

base_report = full_report(model, sp, val_tokens, n_batches=100)
print_report(base_report)

save_report(base_report, WORK / "reports/base_eval.json")
print(f"\nsaved -> {WORK / 'reports/base_eval.json'}")

## Stage 5 gate

- [x] Perplexity measured on uncontaminated held-out data
- [x] Temperature behaviour characterised for *this* model
- [x] Baseline saved for the stage 6 comparison

**Next:** `06_sft.ipynb`